# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests
import pandas as pd
import time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start,
        'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    last_exception = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            # Retry statuses
            if r.status_code not in RETRY_STATUSES:
                r.raise_for_status()
                raise RuntimeError(f"Request failed with status: {r.status_code}")

            last_exception = RuntimeError(f"Retryable status code {r.status_code}")
        except requests.RequestException as e:
            last_exception = e
            print(f"Request failed: {e}")

        # backoff
        if attempt < tries - 1:
            delay = 2 ** attempt
            print(f"Attempt {attempt + 1} failed ({last_exception}). Retrying in {delay}s...")
            time.sleep(delay)

    raise last_exception

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False
try:
    wx = get_weather('2026-09-05', '2026-10-03')
except Exception as e:
  print(e)
  print(f"API call failed: {e}")
  used_fallback = True
pass

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})
# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx['source'] = 'open-meteo'
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
print(wx)

          time  temperature_2m_max  precipitation_sum      source
0   2026-09-05                29.4                2.5  open-meteo
1   2026-09-06                27.2                1.6  open-meteo
2   2026-09-07                27.7                0.0  open-meteo
3   2026-09-08                29.4                0.0  open-meteo
4   2026-09-09                30.9                0.0  open-meteo
5   2026-09-10                34.2                3.2  open-meteo
6   2026-09-11                28.0               10.4  open-meteo
7   2026-09-12                27.5               21.5  open-meteo
8   2026-09-13                31.3                5.8  open-meteo
9   2026-09-14                28.7                0.3  open-meteo
10  2026-09-15                25.8                0.0  open-meteo
11  2026-09-16                29.7                0.0  open-meteo
12  2026-09-17                33.7                0.6  open-meteo
13  2026-09-18                30.0                1.6  open-meteo
14  2026-0

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
target_dates = ['2026-09-05', '2026-10-03']

# If 'time' is missing but 'date' is present, we have already processed the columns
if 'time' in wx.columns:
    wx = wx[wx['time'].isin(target_dates)].copy()
    wx = pd.DataFrame({
        'date': pd.to_datetime(wx['time']),
        'temp_max': wx['temperature_2m_max'],
        'precip_mm': wx['precipitation_sum'],
        'source': wx['source']
    })

wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
#October 3rd was wetter by 12.2mm.

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
print(f"Endpoint: {BASE}")
print(f"Coordinates: 38.03, -78.51")
print(f"Date range: 2026-09-05 to 2026-10-03")
import datetime
print(f"Pulled at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Data source type: {'fallback sample' if used_fallback else 'real API data'}")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range: 2026-09-05 to 2026-10-03
Pulled at: 2026-10-12 01:32:47
Data source type: real API data


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

The chart would should 200 days of sales, but the weather column would either be empty or undefined. The code may interpret the lack of data as just zero precipitation and would falsely correlate sales with false data. Therefore, nobody would catch it because the sales would just look normal.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:

target_dates = ['2026-09-05', '2026-10-03']

# getting data for Charlottesville and Orlando
df_cville = get_weather('2026-09-05', '2026-10-03', lat=38.03, lon=-78.51)
df_cville['city'] = 'Charlottesville'

df_orlando = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
df_orlando['city'] = 'Orlando'

# Combining into one DataFrame
combined_raw = pd.concat([df_cville, df_orlando], ignore_index=True)

# Filtering for target dates
combined_filtered = combined_raw[combined_raw['time'].isin(target_dates)].copy()

wx_multi = pd.DataFrame({
    'date': pd.to_datetime(combined_filtered['time']),
    'city': combined_filtered['city'],
    'temp_max': combined_filtered['temperature_2m_max'],
    'precip_mm': combined_filtered['precipitation_sum']
})

wx_multi

,date,city,temp_max,precip_mm
0,2026-09-05,Charlottesville,29.4,2.5
28,2026-10-03,Charlottesville,22.0,14.7
29,2026-09-05,Orlando,32.1,3.9
57,2026-10-03,Orlando,30.4,14.5
